In [27]:
!pip install faiss-cpu



In [2]:
!pip install ipywidgets

In [3]:
pip install streamlit


In [4]:
!pip install textblob


In [5]:
pip install pypdf


In [6]:
!pip install groq

In [7]:
!pip install ollama

import ollama

In [8]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from textblob import TextBlob
import streamlit as st
import ipywidgets as widgets
from IPython.display import display
import numpy as np
from textblob import TextBlob
from sentence_transformers import CrossEncoder
from groq import Groq


In [9]:
#####Categories for questions-Helps the model perform better by searching the pdf where exactly it should,

categories = [
    "Application",
    "Important Dates and Deadlines",
    "Orientation and Pre-semester Programme",
    "Buddy Service",
    "Courses",
    "Accommodation",
    "Entry and Residence Regulations",
    "Health Insurance",
    "Arrival to Linz",
    "Public Transport",
    "Contact us"
]

In [10]:
def read_doc():
  """
  Reads the pdf file and returns the text extracted from pages.
  """
  reader = PdfReader("Handbook_for_exchange_students.pdf")
  text = ""

  for page in reader.pages:
      text += page.extract_text() + "\n"
  return text
text = read_doc()

In [11]:
def text_chunker(chunk_size=250,overlap=50):
  """
  Splits the text into chunks of a specified size with an overlap.

  """


  chunks = []

  start = 0

  while start < len(text):
    end = start + chunk_size
    chunks.append(text[start:end])
    start = end - overlap

  return chunks
chunks = text_chunker()
print(len(chunks))

174


In [12]:
#################Sentence Embeddings
def db_encode(model_name):

  """
  Encodes the text chunks using the specified model and adds them to a FAISS index db.

  """

  model = SentenceTransformer(model_name)
  embeddings = model.encode(chunks)
  dimensions = embeddings.shape[1]
  index = faiss.IndexFlatL2(dimensions)
  index.add(np.array(embeddings))

  return model,embeddings,index
model,embeddings,index = db_encode("all-MiniLM-L6-v2")



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
#Reranker used to enhance the model performance by reranking the top 10 chunks.
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
def correct_query(query):
  """
  Corrects the query in case the user tyeps it wrong.
  (Sometimes it corrects already corrrect words which then enter the model wrong)
  The word "Visa" is corrected to "Isa" which is wrong.
  Overall I concluded it enhances the model performance although it has some negatives.
  """
  corrected_query = TextBlob(query).correct()
  return corrected_query.raw


def run_rag(question, category):
    """
    Runs Rag

    """

    user_query = correct_query(question)

    search_query = f"{category}: {user_query}"

    query_embedding = model.encode([search_query])

    distance, indices = index.search(np.array(query_embedding), 10)

    relevant_chunks = [chunks[i] for i in indices[0]]

    pairs = [(search_query, chunk) for chunk in relevant_chunks]

    scores = reranker.predict(pairs)

    reranked = sorted(zip(relevant_chunks, scores), key=lambda x: x[1], reverse=True)

    top_chunks = [chunk for chunk, score in reranked[:3]]

    context = "\n\n".join(top_chunks)

    prompt = f"""
You are an assistant that answers questions using the provided documents.

Rules:
- Use ONLY the information from the context.
- If the answer is in the documents, give a clear answer.

If the answer is NOT found:
- Inform the user the information is not available in the documents.
- Suggest they choose one of the document titles and ask again.
- Provide this contact email: international-stuff@jku.at

Context:
{context}

Question:
{user_query}

Answer:
"""
    client = Groq(api_key="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")
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content



In [24]:
# dropdown
category_dropdown = widgets.Dropdown(
    options=categories,
    description="Category:"
)

# text input
question_input = widgets.Text(
    placeholder="Ask your question here",
    description="Question:"
)

# button
ask_button = widgets.Button(description="Ask",button_style="success")

# output area
output = widgets.Output()



In [25]:
def on_button_clicked(b):

    """
    Just User Interface
    """

    with output:

        output.clear_output()

        category = category_dropdown.value
        question = question_input.value

        if question.strip() == "":
            print("Please enter a question.")
            return

        print("Searching documents...\n")

        answer = run_rag(question, category)

        print("Answer:\n")
        print(answer)

display(category_dropdown, question_input, ask_button, output)
ask_button.on_click(on_button_clicked)

Dropdown(description='Category:', options=('Application', 'Important Dates and Deadlines', 'Orientation and Pr…

Text(value='', description='Question:', placeholder='Ask your question here')

Button(button_style='success', description='Ask', style=ButtonStyle())

Output()